# testmon (pytest-testmon) — Selective Test Execution
Install: `pip install pytest-testmon` | CLI: `pytest --testmon`

In [1]:
import testmon
print(dir(testmon))

['TESTMON_VERSION', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [2]:
import pkgutil, importlib
for finder, name, ispkg in pkgutil.walk_packages(testmon.__path__, testmon.__name__ + '.'):
    try:
        mod = importlib.import_module(name)
        print(name, '->', dir(mod))
    except Exception as e:
        print(name, '-> ERROR:', e)

testmon.common -> ['DepsNOutcomes', 'Dict', 'Duration', 'Failed', 'FileFp', 'List', 'Path', 'TestExecutions', 'TestFileFps', 'TestName', 'TypedDict', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'drop_patch_version', 'dummy', 'get_logger', 'get_system_packages', 'get_system_packages_raw', 'git_current_branch', 'git_current_head', 'git_path', 'importlib', 'logger', 'logging', 'os', 're']
testmon.configure -> ['TmConf', 'Tracer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_formulate_deactivation', '_get_nocollect_reasons', '_get_noselect_reasons', '_get_notestmon_reasons', '_header_collect_select', '_is_coverage', '_is_debugger', '_is_dogfooding', 'dataclass', 'header_collect_select', 're', 'sys']
testmon.db -> ['ChangedFileData', 'DATA_VERSION', 'DB', 'TestExecutions', 'TestmonDbException', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name

In [3]:
# Run pytest --testmon and capture raw output
import subprocess, tempfile, os

proj = tempfile.mkdtemp()

with open(os.path.join(proj, 'calculator.py'), 'w') as f:
    f.write('''
def add(a, b): return a + b
def multiply(a, b): return a * b
def divide(a, b):
    if b == 0: raise ZeroDivisionError
    return a / b
''')

with open(os.path.join(proj, 'test_calc.py'), 'w') as f:
    f.write('''
from calculator import add, multiply, divide
import pytest
def test_add(): assert add(2, 3) == 5
def test_multiply(): assert multiply(3, 4) == 12
def test_divide(): assert divide(10, 2) == 5.0
def test_divide_zero():
    with pytest.raises(ZeroDivisionError): divide(1, 0)
''')

r = subprocess.run(
    ['pytest', '--testmon', '-v', '--tb=short'],
    capture_output=True, text=True, cwd=proj, timeout=60
)
print('STDOUT:')
print(r.stdout)
print('STDERR:')
print(r.stderr)
print('RETURNCODE:', r.returncode)

STDOUT:
============================= test session starts =============================
platform win32 -- Python 3.11.9, pytest-7.4.0, pluggy-1.6.0 -- C:\Users\vinod\AppData\Local\Programs\Python\Python311\python.exe
cachedir: .pytest_cache
hypothesis profile 'default'
testmon: new DB, environment: default
We'd like to hear from testmon users! Please go to https://testmon.org/survey to leave feedback.
rootdir: C:\Users\vinod\AppData\Local\Temp\tmpvepk0gun
plugins: anyio-4.11.0, hypothesis-6.151.9, langsmith-0.4.41, pyfakefs-5.9.3, asyncio-0.21.2, cov-7.0.0, testmon-2.2.0
asyncio: mode=Mode.STRICT
collecting ... collected 4 items

test_calc.py::test_add PASSED                                            [ 25%]
test_calc.py::test_multiply PASSED                                       [ 50%]
test_calc.py::test_divide PASSED                                         [ 75%]
test_calc.py::test_divide_zero PASSED                                    [100%]

============================== 4 passed i

In [4]:
# Raw .testmondata SQLite contents
import sqlite3, os, json

db_path = os.path.join(proj, '.testmondata')
if os.path.exists(db_path):
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    # List all tables
    tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print('Tables:', tables)
    for (tbl,) in tables:
        print(f'\n--- {tbl} ---')
        cols = [d[0] for d in cur.execute(f'PRAGMA table_info({tbl})').fetchall()]
        print('Columns:', cols)
        rows = cur.execute(f'SELECT * FROM {tbl}').fetchall()
        for row in rows:
            print(dict(zip(cols, row)))
    con.close()
else:
    print('No .testmondata — run tests first')

Tables: [('metadata',), ('environment',), ('test_execution',), ('file_fp',), ('test_execution_file_fp',), ('suite_execution_file_fsha',)]

--- metadata ---
Columns: [0, 1]
{0: 'None:last_survey_notification_date', 1: '"2026-03-26"'}
{0: 'None:time_saved', 1: '0'}
{0: 'None:time_all', 1: '0.0107434000055946'}
{0: 'None:tests_saved', 1: '0'}
{0: 'None:tests_all', 1: '4'}

--- environment ---
Columns: [0, 1, 2, 3]
{0: 1, 1: 'default', 2: 'CacheControl 0.14, Farama-Notifications 0.0, Flask 3.1, GitPython 3.1, Jinja2 3.1, MarkupSafe 3.0, PyDictionary 2.0, PyDriller 2.9, PyJWT 2.12, PyPDF2 3.0, PyPika 0.48, PyYAML 6.0, Pygments 2.19, SQLAlchemy 2.0, Send2Trash 2.1, Werkzeug 3.1, aiohappyeyeballs 2.6, aiohttp 3.13, aiosignal 1.4, annotated-doc 0.0, annotated-types 0.7, anybadge 1.16, anyio 4.11, argon2-cffi 25.1, argon2-cffi-bindings 25.1, arrow 1.4, astroid 4.0, asttokens 2.4, async-lru 2.3, attrs 25.4, babel 2.18, backoff 2.2, bandit 1.9, bcrypt 5.0, beautifulsoup4 4.14, beniget 0.5, black 

In [5]:
# Raw TestmonData API output
from testmon.testmon_core import TestmonData
import json

db_path = os.path.join(proj, '.testmondata')
if os.path.exists(db_path):
    data = TestmonData(proj)
    print('TestmonData attrs:', dir(data))
    print()
    print('all_nodes:', list(getattr(data, 'all_nodes', [])))
    print('stable_nodeids:', list(getattr(data, 'stable_nodeids', [])))
    print('unstable_nodeids:', list(getattr(data, 'unstable_nodeids', [])))
else:
    print('No .testmondata found')

TestmonData attrs: ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__test__', '__weakref__', '_init_for_local_run', '_init_for_worker', 'all_files', 'all_tests', 'avg_durations', 'close_connection', 'db', 'determine_stable', 'environment', 'exec_id', 'failing_tests', 'fetch_saving_stats', 'files_of_interest', 'for_local_run', 'for_worker', 'get_tests_fingerprints', 'new_db', 'rootdir', 'save_test_execution_file_fps', 'source_tree', 'stable_files', 'stable_test_names', 'sync_db_fs_tests', 'system_packages_change', 'unstable_files', 'unstable_test_names']

all_nodes: []
stable_nodeids: []
unstable_nodeids: []
